# Notebook 22 — End-to-End Agent Integration and Evaluation Harness

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

Notebook 21 produced the Kaggle-facing battle agent.

Notebook 22 validates that the entire production chain can be loaded, exercised,
measured, and monitored as one integrated system.

## Integrated pipeline

```text
Official observation or adapted snapshot
                ↓
KaggleBattleAgent
                ↓
BattlePolicy
                ↓
Observation adapter
                ↓
BattleSnapshot
                ↓
Battle and player feature extraction
                ↓
Legal-action ranking
                ↓
Official option indices

## Notebook Objectives
Load the Notebook 21 production agent.
Verify its dependencies and public interface.
Reuse the validated 60-card deck.
Test deck-request handling.
Test action-selection handling.
Test safe fallback behavior.
Test repeated inference.
Benchmark latency and throughput.
Capture decision and error statistics.
Build an evaluation harness for later replay and self-play experiments.
Export reusable integration code.
Save Notebook 22 through PowerShell.

## Cell 2 — Imports

In [1]:
from __future__ import annotations

import sys
import time
import types
import uuid

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

print("Python:", sys.version)
print("Working directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


## Cell 3 — Locate the project

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    required_markers = {
        "notebooks",
        "scripts",
        "src",
        "data",
    }

    for candidate in [current, *current.parents]:
        found = {
            marker
            for marker in required_markers
            if (candidate / marker).exists()
        }

        if len(found) >= 3:
            return candidate

    if current.name.casefold() == "notebooks":
        return current.parent

    return current


PROJECT_ROOT = find_project_root()

NOTEBOOK21_EXPORT = (
    PROJECT_ROOT
    / "scripts"
    / "21_kaggle_battle_agent.py"
)

NOTEBOOK22_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook22"
)

NOTEBOOK22_PACKAGE_DIR = (
    PROJECT_ROOT
    / "src"
    / "evaluation_harness"
)

for directory in [
    NOTEBOOK22_REPORT_DIR,
    NOTEBOOK22_PACKAGE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook 21 export:", NOTEBOOK21_EXPORT)
print("Notebook 22 reports:", NOTEBOOK22_REPORT_DIR)
print("Evaluation package:", NOTEBOOK22_PACKAGE_DIR)

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 21 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\21_kaggle_battle_agent.py
Notebook 22 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook22
Evaluation package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\evaluation_harness


## Cell 4 — Confirm Notebook 21 export exists

In [3]:
if not NOTEBOOK21_EXPORT.is_file():
    raise FileNotFoundError(
        "Notebook 21 export was not found:\n"
        f"{NOTEBOOK21_EXPORT}"
    )

print("Notebook 21 export located.")
print("Size:", NOTEBOOK21_EXPORT.stat().st_size, "bytes")

Notebook 21 export located.
Size: 15235 bytes


## Cell 5 — Load Notebook 21 safely

In [4]:
source_21 = NOTEBOOK21_EXPORT.read_text(
    encoding="utf-8-sig"
)

source_21_lines = source_21.splitlines()

# Remove exported future imports and restore exactly one.
cleaned_21_lines = [
    line
    for line in source_21_lines
    if line.strip() != "from __future__ import annotations"
]

cleaned_21_source = (
    "from __future__ import annotations\n"
    + "\n".join(cleaned_21_lines)
)

# Patch dynamic modules created from specs so Python 3.13
# dataclass processing can resolve module namespaces.
module_variable_names = [
    "notebook18",
    "notebook19",
    "notebook20",
]

for variable_name in module_variable_names:
    spec_name = {
        "notebook18": "spec18",
        "notebook19": "spec19",
        "notebook20": "spec20",
    }[variable_name]

    original = (
        f"{variable_name} = "
        f"importlib.util.module_from_spec({spec_name})\n"
        f"{spec_name}.loader.exec_module({variable_name})"
    )

    replacement = (
        f"{variable_name} = "
        f"importlib.util.module_from_spec({spec_name})\n"
        f"sys.modules[{spec_name}.name] = {variable_name}\n"
        f"{spec_name}.loader.exec_module({variable_name})"
    )

    cleaned_21_source = cleaned_21_source.replace(
        original,
        replacement,
    )

module_name_21 = (
    "notebook21_kaggle_agent_"
    + uuid.uuid4().hex
)

notebook21 = types.ModuleType(module_name_21)
notebook21.__file__ = str(NOTEBOOK21_EXPORT)
notebook21.__package__ = ""

sys.modules[module_name_21] = notebook21

compiled_21 = compile(
    cleaned_21_source,
    str(NOTEBOOK21_EXPORT),
    "exec",
)

exec(
    compiled_21,
    notebook21.__dict__,
)

print("\nNotebook 21 loaded successfully.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 20 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\20_policy_engine.py
Kaggle agent package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\kaggle_agent
Notebook 21 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook21
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Notebook 19 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battl

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


...

SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

,rank,option_index,option_type,semantic_label,score,reasons
0,1,0,ATTACK,Attack with Mega Lucario ex,259.913,base=100.0 | attack_bonus=150.0 | active_energ...
1,2,1,END,End Turn,-25.000,base=0.0 | end_turn_penalty=-25.0


[OK] player_feature_extraction
[OK] battle_feature_extraction
[OK] action_feature_extraction
[OK] legal_action_ranking
[OK] best_option_index

Notebook 19 validation passed.
Notebook 19 loaded.
Production functions imported.
True
True
True
True
True

Notebook dependencies verified.
Repository size: 1267
Official CardData lookup size: 1267

Policy dependencies loaded.
BattlePolicy created.
Debug mode: True
BattlePolicy created successfully.
Safe BattlePolicy created successfully.
Option index: 0
Fallback used: True
Reason: Observation adaptation failed: AttributeError: 'NoneType' object has no attribute 'current'

Fallback behavior passed.
Synthetic snapshot loaded.
Turn: 3
Legal options: 2

Synthetic snapshot validation passed.
BattlePolicy Decision
Chosen option: 0
Fallback used: False
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex

Ranked actions:
 1. index=0   type=ATTACK     score=  259.913 Attack with Mega Lucario ex
 2. index=1   type=END        score=

## Cell 6 — Inspect required production objects

In [5]:
REQUIRED_NOTEBOOK21_OBJECTS = [
    "AgentStats",
    "KaggleBattleAgent",
    "kaggle_agent",
    "production_agent",
    "policy",
    "deck_ids",
    "sample_snapshot",
]

missing_objects = [
    name
    for name in REQUIRED_NOTEBOOK21_OBJECTS
    if not hasattr(notebook21, name)
]

for name in REQUIRED_NOTEBOOK21_OBJECTS:
    print(
        f"{'[OK]' if hasattr(notebook21, name) else '[MISSING]'} "
        f"{name}"
    )

if missing_objects:
    raise AttributeError(
        "Notebook 21 export is missing:\n"
        + "\n".join(missing_objects)
    )

print("\nNotebook 21 production interface verified.")

[OK] AgentStats
[OK] KaggleBattleAgent
[OK] kaggle_agent
[OK] production_agent
[OK] policy
[OK] deck_ids
[OK] sample_snapshot

Notebook 21 production interface verified.


## Cell 7 — Retrieve the production objects

In [6]:
AgentStats = notebook21.AgentStats
KaggleBattleAgent = notebook21.KaggleBattleAgent

kaggle_agent = notebook21.kaggle_agent
policy = notebook21.policy

production_agent = notebook21.production_agent

sample_snapshot = notebook21.sample_snapshot
deck_ids = notebook21.deck_ids

print("Production objects loaded.")

print("Deck size:", len(deck_ids))
print("Agent type:", type(kaggle_agent).__name__)
print("Policy type:", type(policy).__name__)

assert len(deck_ids) == 60

print("\nNotebook 21 objects imported successfully.")

Production objects loaded.
Deck size: 60
Agent type: KaggleBattleAgent
Policy type: BattlePolicy

Notebook 21 objects imported successfully.


## Cell 8 — Verify the production agent interface

In [7]:
required_methods = [
    "choose",
    "stats",
]

missing = [
    method
    for method in required_methods
    if not hasattr(kaggle_agent, method)
]

for method in required_methods:
    print(
        "[OK]" if hasattr(kaggle_agent, method)
        else "[MISSING]",
        method,
    )

assert not missing

print("\nProduction agent interface verified.")

[OK] choose
[OK] stats

Production agent interface verified.


## Cell 9 — Test deck submission again

In [8]:
class DeckRequest:
    select = None

deck_request = DeckRequest()

returned_deck = kaggle_agent.choose(deck_request)

print("Returned cards:", len(returned_deck))

assert isinstance(returned_deck, list)
assert len(returned_deck) == 60
assert returned_deck == list(deck_ids)

print("\nDeck submission verified.")

Returned cards: 60

Deck submission verified.


## Cell 10 — Test action selection again

In [9]:
chosen = kaggle_agent.choose(sample_snapshot)

print("Chosen:", chosen)

assert chosen == [0]

print("\nAction selection verified.")

Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Chosen: [0]

Action selection verified.


# Cell 11 — Stress test the production agent

In [10]:
results = []

for i in range(25):
    result = kaggle_agent.choose(sample_snapshot)

    print(
        f"Run {i+1:02d}:",
        result,
    )

    results.append(result)

assert all(r == [0] for r in results)

print()
print("Repeated inference passed.")
print("Calls:", kaggle_agent.calls)
print("Errors:", kaggle_agent.errors)
print("Fallbacks:", kaggle_agent.fallbacks)

Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 01: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 02: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 03: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 04: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 05: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 06: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 07: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 08: [0]
Agent result: [0]
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex
Run 09: [0]
Agent result: [0]
Reason: Selected highest-scoring lega

# Cell 12 — Validate AgentStats

In [11]:
stats = kaggle_agent.stats()

print(stats)
print("Calls:", stats.calls)
print("Errors:", stats.errors)
print("Fallbacks:", stats.fallbacks)
print("Total seconds:", stats.total_seconds)
print("Average seconds:", stats.average_seconds)

assert stats.calls >= 29
assert stats.errors == 0
assert stats.fallbacks == 0
assert stats.total_seconds >= 0.0
assert stats.average_seconds >= 0.0

print("\nAgentStats validation passed.")

AgentStats(calls=29, errors=0, fallbacks=0, total_seconds=0.003190099996572826)
Calls: 29
Errors: 0
Fallbacks: 0
Total seconds: 0.003190099996572826
Average seconds: 0.00011000344815768366

AgentStats validation passed.


# Cell 13 — Build the evaluation harness

In [12]:
@dataclass(frozen=True)
class EvaluationRecord:
    run_index: int
    result: tuple[int, ...]
    elapsed_seconds: float
    valid: bool


@dataclass
class AgentEvaluationHarness:
    agent: KaggleBattleAgent

    def evaluate(
        self,
        observation: Any,
        runs: int = 100,
        expected: Sequence[int] | None = None,
    ) -> list[EvaluationRecord]:
        records: list[EvaluationRecord] = []

        for run_index in range(1, runs + 1):
            started = time.perf_counter()

            result = self.agent.choose(observation)

            elapsed = time.perf_counter() - started

            valid = (
                True
                if expected is None
                else list(result) == list(expected)
            )

            records.append(
                EvaluationRecord(
                    run_index=run_index,
                    result=tuple(result),
                    elapsed_seconds=elapsed,
                    valid=valid,
                )
            )

        return records

## Cell 14 — Run the harness

In [13]:
kaggle_agent.debug = False

harness = AgentEvaluationHarness(
    agent=kaggle_agent
)

evaluation_records = harness.evaluate(
    sample_snapshot,
    runs=100,
    expected=[0],
)

valid_count = sum(
    record.valid
    for record in evaluation_records
)

elapsed_values = [
    record.elapsed_seconds
    for record in evaluation_records
]

print("Runs:", len(evaluation_records))
print("Valid:", valid_count)
print("Invalid:", len(evaluation_records) - valid_count)
print("Minimum seconds:", min(elapsed_values))
print("Maximum seconds:", max(elapsed_values))
print(
    "Average seconds:",
    sum(elapsed_values) / len(elapsed_values),
)

assert valid_count == 100

print("\nEvaluation harness passed.")

Runs: 100
Valid: 100
Invalid: 0
Minimum seconds: 3.9200000173877925e-05
Maximum seconds: 0.0001143000008596573
Average seconds: 4.279999975551618e-05

Evaluation harness passed.


### Above my Benchmark result is interpreted as: 

| Metric  | Result      | Status       |
| ------- | ----------- | -----------  |
| Runs    | 100         | ✅           |
| Valid   | 100         | ✅           |
| Invalid | 0           | ✅           |
| Minimum | 39.2 μs     | ✅           |
| Maximum | 114.3 μs    | ✅           |
| Average | **42.8 μs** | ⭐ Excellent |

##### An average inference time of ~43 microseconds for a policy decision is extremely fast and means your wrapper is adding virtually no overhead.


## Cell 15 — Validate the complete Notebook 22

In [14]:
notebook22_checks = {
    "production_interface": True,
    "stress_test": len(results) == 25,
    "stress_errors": kaggle_agent.errors == 0,
    "stress_fallbacks": kaggle_agent.fallbacks == 0,
    "agent_stats": stats.calls >= 29,
    "evaluation_runs": len(evaluation_records) == 100,
    "evaluation_valid": valid_count == 100,
    "evaluation_invalid": len(evaluation_records) - valid_count == 0,
}

for name, passed in notebook22_checks.items():
    print(
        f"{'[OK]' if passed else '[FAIL]'} {name}"
    )

assert all(notebook22_checks.values())

print("\nNotebook 22 validation passed.")

[OK] production_interface
[OK] stress_test
[OK] stress_errors
[OK] stress_fallbacks
[OK] agent_stats
[OK] evaluation_runs
[OK] evaluation_valid
[OK] evaluation_invalid

Notebook 22 validation passed.


## Cell 16 — Final Summary

In [16]:
print("=" * 72)
print("Notebook 22 — Production Evaluation")
print("=" * 72)

repository = notebook21.repository

official_card_data_by_id = (
    notebook21.official_card_data_by_id
)

print("Repository cards:", len(repository))
print(
    "Official cards:",
    len(official_card_data_by_id),
)
print("Deck size:", len(deck_ids))

print()

print("Policy calls:", kaggle_agent.calls)
print("Policy errors:", kaggle_agent.errors)
print("Policy fallbacks:", kaggle_agent.fallbacks)

print()

print("Stress test:", len(results))
print(
    "Evaluation runs:",
    len(evaluation_records),
)

average_inference = (
    sum(elapsed_values)
    / len(elapsed_values)
)

print("Average inference:", average_inference)

print()

print("NOTEBOOK 22 COMPLETED SUCCESSFULLY")
print("Ready for PowerShell export.")

Notebook 22 — Production Evaluation
Repository cards: 1267
Official cards: 1267
Deck size: 60

Policy calls: 129
Policy errors: 0
Policy fallbacks: 0

Stress test: 25
Evaluation runs: 100
Average inference: 4.279999975551618e-05

NOTEBOOK 22 COMPLETED SUCCESSFULLY
Ready for PowerShell export.


## Above Final Validation Summary 

| Check                               | Status                         |
| ----------------------------------- | ------------------------------ |
| Notebook 21 imported successfully   | ✅                              |
| Repository loaded (1267 cards)      | ✅                              |
| Official lookup loaded (1267 cards) | ✅                              |
| KaggleBattleAgent instantiated      | ✅                              |
| Deck submission                     | ✅                              |
| Action selection                    | ✅                              |
| Stress test (25/25)                 | ✅                              |
| Evaluation harness (100/100)        | ✅                              |
| Policy errors                       | **0** ✅                        |
| Policy fallbacks                    | **0** ✅                        |
| Average inference                   | **4.28 × 10⁻⁵ s (≈42.8 μs)** ✅ |
| Notebook 22 completed               | ✅                              |


##### An average inference time of about 43 microseconds indicates the evaluation wrapper is extremely lightweight.